# 00 · Environment and data audit

**Series:** Early-Exit GNN for clinical risk stratification on MIMIC-IV.

This notebook establishes what we are running on and what data is actually
present, before any modelling claim depends on it. It is deliberately the
cheapest notebook in the series: it reads no event tables.

It answers three questions:

1. **What compute do we have?** Determines whether the 42 GB `chartevents`
   scan and the 47k-node graph are feasible in one session.
2. **Which MIMIC-IV tables are on disk?** The ETL degrades gracefully — a
   missing table means a missing feature block, not a crash — so "which blocks
   are even possible" is a property of the mount, and needs recording.
3. **Do the schemas match what the ETL expects?** Column names in MIMIC-IV
   v3.1 differ from v2.x in places; catching that here costs seconds, catching
   it inside a 40-minute scan does not.

In [1]:
exec(open("/kaggle/working/clinical-eegnn/notebooks/src/_bootstrap.py").read())

import platform, subprocess, importlib

print("python  ", sys.version.split()[0], "|", platform.platform())
print("cwd     ", os.getcwd())

for m in ["numpy", "pandas", "duckdb", "torch", "torch_geometric", "sklearn",
          "xgboost", "shap", "pyarrow", "matplotlib"]:
    try:
        print("  %-18s %s" % (m, getattr(importlib.import_module(m), "__version__", "?")))
    except ImportError:
        print("  %-18s MISSING" % m)

python   3.12.13 | Linux-6.12.90+-x86_64-with-glibc2.35
cwd      /kaggle/working/clinical-eegnn
  numpy              2.0.2
  pandas             2.3.3
  duckdb             1.3.2


  torch              2.10.0+cu128
  torch_geometric    MISSING


  sklearn            1.6.1


  xgboost            3.2.0


  shap               0.51.0
  pyarrow            24.0.0
  matplotlib         3.10.0


## Compute

`duckdb` is the load-bearing dependency here. The event aggregation in
`src/mimic_iv.py` scans `labevents` (18 GB) and `chartevents` (42 GB) with
predicate pushdown and constant memory. Without it the code falls back to a
chunked pandas reader that works but is far slower.

A GPU matters less than it looks: the model is small and the cohort is a single
graph. Where it does pay is kNN construction, which is one big matmul.

In [2]:
import torch
print("cuda available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  gpu{i}: {p.name}  {p.total_memory/1e9:.1f} GB")
print("cpus:", os.cpu_count())
print("ram :", round(int(open("/proc/meminfo").readline().split()[1]) / 1e6, 1), "GB")
print(subprocess.run(["df", "-h", "/kaggle/working"], capture_output=True, text=True).stdout)

cuda available: True


  gpu0: Tesla T4  15.6 GB
  gpu1: Tesla T4  15.6 GB
cpus: 4
ram : 32.9 GB
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  388K   20G   1% /kaggle/working



## Install `torch_geometric`

Not in the Kaggle base image. The model uses only `MessagePassing`, `GCNConv`
and `GATConv`, so the plain wheel is enough — none of the compiled
`torch-scatter` / `torch-sparse` extras are required.

In [3]:
try:
    import torch_geometric
    print("already present:", torch_geometric.__version__)
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"],
                   check=True)
    import torch_geometric
    print("installed:", torch_geometric.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.1 MB/s eta 0:00:00


installed: 2.8.0.post1


## Dataset inventory

The Kaggle mirror ships PhysioNet's layout verbatim. `find_root` locates the
tree by looking for a directory containing `hosp/`, so it tolerates the extra
nesting level Kaggle adds.

In [4]:
from src import mimic_iv

root = mimic_iv.find_root(RAW)
print("root:", root)
assert root is not None, f"no MIMIC-IV tree under {RAW}"

rows = []
for sub in ("hosp", "icu"):
    d = os.path.join(root, sub)
    for f in sorted(os.listdir(d)):
        rows.append({"table": f"{sub}/{f}", "GB": os.path.getsize(os.path.join(d, f)) / 1e9})
inv = pd.DataFrame(rows).sort_values("GB", ascending=False)
print(f"{len(inv)} tables, {inv['GB'].sum():.1f} GB total\n")
inv.head(12).to_string(index=False)

root: /kaggle/input/datasets/akshaybe/updated-mimic-iv/mimic-iv-3.1
31 tables, 97.2 GB total



'                      table        GB\n        icu/chartevents.csv 41.935806\n         hosp/labevents.csv 18.402852\n       hosp/emar_detail.csv  8.682184\n              hosp/emar.csv  6.248519\n               hosp/poe.csv  5.101087\n          hosp/pharmacy.csv  3.975713\n     hosp/prescriptions.csv  3.488170\n        icu/inputevents.csv  2.868896\n   icu/ingredientevents.csv  2.472156\n     icu/datetimeevents.csv  1.092343\nhosp/microbiologyevents.csv  0.909240\n       icu/outputevents.csv  0.462494'

## Which tables the pipeline actually reads

Only a small fraction of that 100 GB is touched. Each feature block names its
source; a block whose source is absent is skipped and simply contributes no
columns.

In [5]:
NEEDED = {
    "hosp/patients.csv":       "cohort: age, sex, date of death",
    "hosp/admissions.csv":     "cohort: admission/discharge times, outcome flag",
    "icu/icustays.csv":        "cohort: ICU window, care unit",
    "hosp/diagnoses_icd.csv":  "block: comorbidity (prior admissions only)",
    "hosp/labevents.csv":      "block: labs",
    "icu/chartevents.csv":     "block: vitals",
    "icu/inputevents.csv":     "block: treatment (drugs, fluids)",
    "icu/procedureevents.csv": "block: treatment (lines, ventilation)",
    "icu/outputevents.csv":    "block: treatment (urine output)",
}
missing = []
for rel, why in NEEDED.items():
    ok = mimic_iv.has_table(root, rel)
    missing += [] if ok else [rel]
    print(f"  {'OK ' if ok else 'GONE'}  {rel:26s} {why}")
print("\nall required tables present" if not missing else f"\nMISSING: {missing}")

  OK   hosp/patients.csv          cohort: age, sex, date of death
  OK   hosp/admissions.csv        cohort: admission/discharge times, outcome flag
  OK   icu/icustays.csv           cohort: ICU window, care unit
  OK   hosp/diagnoses_icd.csv     block: comorbidity (prior admissions only)
  OK   hosp/labevents.csv         block: labs
  OK   icu/chartevents.csv        block: vitals
  OK   icu/inputevents.csv        block: treatment (drugs, fluids)
  OK   icu/procedureevents.csv    block: treatment (lines, ventilation)
  OK   icu/outputevents.csv       block: treatment (urine output)

all required tables present


## Schema check

The ETL reads specific columns by name. This reads only the header plus a
handful of rows from each table — cheap even for the 42 GB files, because
`nrows` stops the reader immediately.

In [6]:
EXPECTED = {
    "hosp/patients.csv":   ["subject_id", "gender", "anchor_age", "anchor_year", "dod"],
    "hosp/admissions.csv": ["subject_id", "hadm_id", "admittime", "dischtime",
                            "deathtime", "admission_type", "admission_location",
                            "insurance", "marital_status", "race",
                            "hospital_expire_flag"],
    "icu/icustays.csv":    ["subject_id", "hadm_id", "stay_id", "first_careunit",
                            "intime", "outtime", "los"],
    "hosp/labevents.csv":  ["hadm_id", "itemid", "charttime", "valuenum"],
    "icu/chartevents.csv": ["stay_id", "itemid", "charttime", "valuenum"],
    "icu/inputevents.csv": ["stay_id", "itemid", "starttime"],
    "icu/outputevents.csv":["stay_id", "itemid", "charttime", "value"],
}
for rel, cols in EXPECTED.items():
    head = pd.read_csv(mimic_iv.table_path(root, rel), nrows=3)
    absent = [c for c in cols if c not in head.columns]
    print(f"  {rel:24s} {len(head.columns):3d} cols  " +
          ("all expected present" if not absent else f"!! MISSING {absent}"))

  hosp/patients.csv          6 cols  all expected present
  hosp/admissions.csv       16 cols  all expected present


  icu/icustays.csv           8 cols  all expected present
  hosp/labevents.csv        16 cols  all expected present


  icu/chartevents.csv       11 cols  all expected present
  icu/inputevents.csv       26 cols  all expected present
  icu/outputevents.csv       9 cols  all expected present


## Concept dictionaries

Features are defined by `itemid` sets in `src/mimic_concepts.py`. A concept
whose itemids never appear would silently produce an all-missing column, so
confirm the ids resolve against the dictionaries.

In [7]:
from src import mimic_concepts as mc

d_labs = pd.read_csv(mimic_iv.table_path(root, "hosp/d_labitems.csv"))
d_items = pd.read_csv(mimic_iv.table_path(root, "icu/d_items.csv"))

for label, items, dic in [("labs", mc.LAB_ITEMS, d_labs),
                          ("chart", mc.CHART_ITEMS, d_items),
                          ("input", mc.INPUT_ITEMS, d_items),
                          ("proc", mc.PROC_ITEMS, d_items)]:
    known = set(dic["itemid"])
    unknown = [i for i in items if i not in known]
    concepts = sorted({v[0] for v in items.values()})
    print(f"  {label:6s} {len(items):3d} itemids -> {len(concepts):2d} concepts"
          f"   unresolved: {unknown if unknown else 'none'}")

urine = set(mc.OUTPUT_URINE_ITEMS) | set(mc.OUTPUT_URINE_NEGATIVE_ITEMS)
print("\nurine itemids:")
print(d_items[d_items["itemid"].isin(urine)][["itemid", "label"]].to_string(index=False))

  labs    28 itemids -> 28 concepts   unresolved: none
  chart   22 itemids -> 14 concepts   unresolved: none
  input   14 itemids ->  9 concepts   unresolved: none
  proc    15 itemids ->  7 concepts   unresolved: none

urine itemids:
 itemid                        label
 226557             R Ureteral Stent
 226558             L Ureteral Stent
 226559                        Foley
 226560                         Void
 226561                  Condom Cath
 226563                   Suprapubic
 226564                R Nephrostomy
 226565                L Nephrostomy
 226567                Straight Cath
 226584                  Ileoconduit
 227488        GU Irrigant Volume In
 227489 GU Irrigant/Urine Volume Out


### On the urine itemids

`227488` ("GU Irrigant Volume In") is fluid *instilled* for bladder irrigation,
not urine produced, and `227489` ("GU Irrigant/Urine Volume Out") returns it
mixed with urine. MIMIC's own `urine_output` concept subtracts the former from
the latter, and `src/mimic_iv._urine_totals` now does the same. Summing both as
output overstates urine for anyone on irrigation — mostly post-urology
patients, i.e. a systematic bias in a specific subgroup rather than noise.

## Audit result

Record the outcome so later notebooks can assert they are running against the
same mount.

In [8]:
audit = {
    "root": root,
    "tables_present": sorted(NEEDED),
    "tables_missing": missing,
    "blocks_possible": ["demographics", "comorbidity"]
                       + (["labs"] if mimic_iv.has_table(root, "hosp/labevents.csv") else [])
                       + (["vitals"] if mimic_iv.has_table(root, "icu/chartevents.csv") else [])
                       + (["treatment"] if mimic_iv.has_table(root, "icu/inputevents.csv") else []),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "cpus": os.cpu_count(),
}
os.makedirs(CACHE, exist_ok=True)
json.dump(audit, open(f"{CACHE}/audit.json", "w"), indent=2)
print(json.dumps(audit, indent=2))

{
  "root": "/kaggle/input/datasets/akshaybe/updated-mimic-iv/mimic-iv-3.1",
  "tables_present": [
    "hosp/admissions.csv",
    "hosp/diagnoses_icd.csv",
    "hosp/labevents.csv",
    "hosp/patients.csv",
    "icu/chartevents.csv",
    "icu/icustays.csv",
    "icu/inputevents.csv",
    "icu/outputevents.csv",
    "icu/procedureevents.csv"
  ],
  "tables_missing": [],
  "blocks_possible": [
    "demographics",
    "comorbidity",
    "labs",
    "vitals",
    "treatment"
  ],
  "gpu": "Tesla T4",
  "cpus": 4
}


**Next:** `01_cohort_extraction` turns these tables into one patient-level
design matrix, and asserts the leakage controls that make the label honest.